# 02b — Default Parameter Experiments
**Tujuan:** Menjalankan eksperimen yang identik dengan notebook 02a Section 14,  
satu-satunya perbedaan adalah **hyperparameter default** (bukan Optuna-tuned).

Hasil notebook ini digunakan untuk **tabel perbandingan Default vs Optuna** yang diminta dosen.

| Aspek | Notebook 02a (Optuna) | Notebook 02b (Default) |
|---|---|---|
| Hyperparameter | Optuna-tuned | **Konservatif/default** |
| Evaluasi | 80/20 temporal split | 80/20 temporal split ✓ |
| Group covariates | Sama | Sama ✓ |
| Windows | [20, 120] | [20, 120] ✓ |
| Horizons | [1, 5, 20] | [1, 5, 20] ✓ |

## Output
- `phase1b_default_results.csv` — 144 baris hasil default params
- Tabel perbandingan parameter Default vs Optuna (untuk paper)

In [2]:
import pandas as pd
import numpy as np
import joblib
import gc
import glob
import traceback
from datetime import datetime
import warnings
warnings.filterwarnings("ignore")

from darts import TimeSeries
from darts.models import RandomForestModel, XGBModel, LightGBMModel
from darts.dataprocessing.transformers import Scaler, Diff
from darts.utils.missing_values import fill_missing_values
from sklearn.ensemble import ExtraTreesRegressor

try:
    from darts.models import SKLearnModel
except ImportError:
    from darts.models import RegressionModel as SKLearnModel

LEVEL_VARS  = ["M2","USDIDR","Coal","Copper","Nickel","Silver","Tin","STI","Gold","WTI","GDP"]
RATE_VARS   = ["BI_Rate","CPI","NPL_Ratio","US_Treasury_10Y"]
WINDOWS     = [20, 120]
HORIZONS    = [1, 5, 20]
TRAIN_RATIO = 0.8

df_merged = joblib.load(sorted(glob.glob("saved_models/df_merged_*.joblib"), reverse=True)[0])
print(f"Data: {df_merged.shape} | {df_merged['date'].min().date()} to {df_merged['date'].max().date()}")

Data: (2443, 17) | 2015-01-02 to 2025-01-31


In [3]:
# Default/konservatif hyperparameters — sama dengan yang dipakai di notebook 02
# (sebelum ada Optuna tuning)
DEFAULT_CONFIGS = {
    "RandomForest": {
        "n_estimators": 300,
        "max_depth":    5,
        "max_features": "sqrt",
        "max_samples":  0.7,
    },
    "ExtraTrees": {
        "n_estimators": 300,
        "max_depth":    5,
        "max_features": "sqrt",
        # bootstrap=False default → max_samples tidak berlaku
    },
    "XGBoost": {
        "n_estimators":     300,
        "max_depth":        5,
        "learning_rate":    0.1,
        "subsample":        0.7,
        "colsample_bytree": 0.7,
        "reg_alpha":        0.01,
        "reg_lambda":       1.0,
    },
    "LightGBM": {
        "n_estimators":     300,
        "max_depth":        5,
        "learning_rate":    0.1,
        "num_leaves":       31,
        "subsample":        0.7,
        "colsample_bytree": 0.7,
        "reg_alpha":        0.01,
        "reg_lambda":       1.0,
    },
}

print("Default hyperparameters:")
for model, params in DEFAULT_CONFIGS.items():
    print(f"  {model}: {params}")

Default hyperparameters:
  RandomForest: {'n_estimators': 300, 'max_depth': 5, 'max_features': 'sqrt', 'max_samples': 0.7}
  ExtraTrees: {'n_estimators': 300, 'max_depth': 5, 'max_features': 'sqrt'}
  XGBoost: {'n_estimators': 300, 'max_depth': 5, 'learning_rate': 0.1, 'subsample': 0.7, 'colsample_bytree': 0.7, 'reg_alpha': 0.01, 'reg_lambda': 1.0}
  LightGBM: {'n_estimators': 300, 'max_depth': 5, 'learning_rate': 0.1, 'num_leaves': 31, 'subsample': 0.7, 'colsample_bytree': 0.7, 'reg_alpha': 0.01, 'reg_lambda': 1.0}


In [4]:
import joblib, pandas as pd

TUNING = joblib.load("saved_models/optuna_tuning_results.joblib")
MODEL_ORDER = ["RandomForest", "ExtraTrees", "XGBoost", "LightGBM"]

# Kumpulkan semua parameter yang muncul
all_params = sorted(set(
    p for r in TUNING.values() for p in r["best_params"]
))

# Bangun tabel: baris = parameter, kolom = model
rows = []
for p in all_params:
    row = {"Parameter": p}
    for m in MODEL_ORDER:
        val = TUNING[m]["best_params"].get(p, "—")
        if isinstance(val, float):
            row[m] = f"{val:.4f}" if val >= 0.01 else f"{val:.2e}"
        else:
            row[m] = str(val)
    rows.append(row)

df_opt_params = pd.DataFrame(rows).set_index("Parameter")

# Highlight: kuning jika berbeda dari default (atau tidak ada di default)
def highlight_vs_default(row):
    styles = []
    param = row.name
    for m in MODEL_ORDER:
        opt_val  = TUNING[m]["best_params"].get(param)
        def_val  = DEFAULT_CONFIGS[m].get(param)
        if opt_val is None:
            styles.append("")          # tidak dipakai model ini
        elif def_val is None or str(opt_val) != str(def_val):
            styles.append("background-color:#fff9c4;font-weight:bold")  # berubah
        else:
            styles.append("")          # sama
    return styles

print("=" * 60)
print("  Hyperparameter Optuna — Hasil Tuning per Model")
print("  (Kuning = berbeda dari nilai default)")
print("=" * 60)

display(
    df_opt_params.style
    .apply(highlight_vs_default, axis=1)
    .set_caption(
        "Optuna best params — RF=RandomForest, ET=ExtraTrees, "
        "XGB=XGBoost, LGB=LightGBM. "
        "Kuning = parameter berubah dari default."
    )
)

  Hyperparameter Optuna — Hasil Tuning per Model
  (Kuning = berbeda dari nilai default)


,RandomForest,ExtraTrees,XGBoost,LightGBM
Parameter,,,,
colsample_bytree,—,—,0.3920,0.3455
learning_rate,—,—,0.0295,0.0117
max_depth,3,5,11,10
max_features,0.7000,sqrt,—,—
max_samples,0.8000,—,—,—
min_child_samples,—,—,—,42
min_child_weight,—,—,8,—
min_samples_leaf,6,3,—,—
min_samples_split,5,9,—,—


In [5]:
# Identik dengan notebook 02a Section 14
GROUP_COVARIATES = {
    "Baseline": [],
    "Screening1": [
        "Silver", "WTI", "Gold", "STI",
        "Coal", "Tin", "NPL_Ratio",
    ],
    "Screening2": [
        "Silver", "WTI", "Gold", "STI",
        "Coal", "Tin", "NPL_Ratio",
        "CPI", "USDIDR", "Nickel",
    ],
    "All_Commodity_STI": [
        "Coal", "Copper", "Nickel", "Silver", "Tin", "Gold", "WTI", "STI",
    ],
    "All_Macro_no_UST": [
        "BI_Rate", "CPI", "M2", "NPL_Ratio", "USDIDR", "GDP",
    ],
    "All_Covariates": [
        "BI_Rate", "CPI", "M2", "NPL_Ratio", "USDIDR", "GDP",
        "Coal", "Copper", "Nickel", "Silver", "Tin", "Gold", "WTI", "STI",
        "US_Treasury_10Y",
    ],
}

total_exp = len(GROUP_COVARIATES) * 4 * len(WINDOWS) * len(HORIZONS)
print("Group Covariates:")
for k, v in GROUP_COVARIATES.items():
    print(f"  {k:<22} ({len(v):>2} vars)")
print(f"\nTotal: {len(GROUP_COVARIATES)} groups × 4 models × {len(WINDOWS)} windows × {len(HORIZONS)} horizons = {total_exp}")

Group Covariates:
  Baseline               ( 0 vars)
  Screening1             ( 7 vars)
  Screening2             (10 vars)
  All_Commodity_STI      ( 8 vars)
  All_Macro_no_UST       ( 6 vars)
  All_Covariates         (15 vars)

Total: 6 groups × 4 models × 2 windows × 3 horizons = 144


In [6]:
def to_series(df, target_col, covariates=None):
    target = TimeSeries.from_dataframe(
        df, time_col="date", value_cols=target_col,
        fill_missing_dates=True, freq="B")
    target = fill_missing_values(target)
    cov = None
    if covariates:
        cov = TimeSeries.from_dataframe(
            df, time_col="date", value_cols=covariates,
            fill_missing_dates=True, freq="B")
        cov = fill_missing_values(cov)
    return target, cov


def build_model_default(model_name, window, horizon, has_covariates):
    """Pakai DEFAULT_CONFIGS — bukan Optuna."""
    params = DEFAULT_CONFIGS[model_name].copy()
    common = {
        "lags": window,
        "lags_past_covariates": window if has_covariates else None,
        "output_chunk_length": horizon,
    }
    if model_name == "RandomForest":
        return RandomForestModel(**common, random_state=42, n_jobs=-1, **params)
    elif model_name == "ExtraTrees":
        return SKLearnModel(**common,
            model=ExtraTreesRegressor(random_state=42, n_jobs=-1, **params))
    elif model_name == "XGBoost":
        return XGBModel(**common, random_state=42, n_jobs=-1, **params)
    elif model_name == "LightGBM":
        return LightGBMModel(**common, random_state=42, n_jobs=-1, verbose=-1, **params)
    raise ValueError(f"Unknown model: {model_name}")


def transform_target(target_ts, split_idx):
    train_ts  = target_ts[:split_idx]
    full_log  = target_ts.map(np.log)
    train_log = train_ts.map(np.log)
    diff = Diff(lags=1)
    train_log_diff = diff.fit_transform(train_log)
    full_log_diff  = diff.transform(full_log)
    scaler = Scaler()
    train_scaled = scaler.fit_transform(train_log_diff)
    full_scaled  = scaler.transform(full_log_diff)
    return train_scaled, full_scaled, scaler


def transform_covariates(cov_ts, split_idx):
    if cov_ts is None:
        return None
    train_cov = cov_ts[:split_idx]
    full_cov  = cov_ts
    cov_cols   = cov_ts.components.tolist()
    level_cols = [c for c in cov_cols if c in LEVEL_VARS]
    rate_cols  = [c for c in cov_cols if c in RATE_VARS]
    parts_train, parts_full = [], []
    if level_cols:
        d = Diff(lags=1)
        parts_train.append(d.fit_transform(train_cov[level_cols].map(np.log)))
        parts_full.append(d.transform(full_cov[level_cols].map(np.log)))
    if rate_cols:
        d = Diff(lags=1)
        parts_train.append(d.fit_transform(train_cov[rate_cols]))
        parts_full.append(d.transform(full_cov[rate_cols]))
    ct, cf = parts_train[0], parts_full[0]
    for pt, pf in zip(parts_train[1:], parts_full[1:]):
        ct = ct.stack(pt); cf = cf.stack(pf)
    cov_scaler = Scaler()
    cov_scaler.fit(ct)
    return cov_scaler.transform(cf)


def inverse_and_metrics(forecast_list, full_ts, scaler, split_idx):
    full_log = full_ts.map(np.log)
    all_dates, all_prices = [], []
    for chunk_scaled in forecast_list:
        chunk_diff = scaler.inverse_transform(chunk_scaled)
        dates = chunk_diff.time_index
        vals  = chunk_diff.values().flatten()
        idx   = full_ts.get_index_at_point(dates[0])
        if idx == 0: continue
        anchor     = full_log[idx-1].values()[0][0]
        log_prices = anchor + np.cumsum(vals)
        all_dates.extend(dates); all_prices.extend(np.exp(log_prices))
    pred_df   = pd.DataFrame({"date": pd.to_datetime(all_dates), "predicted": all_prices})
    actual_df = full_ts.to_dataframe().reset_index()
    actual_df.columns = ["date", "actual"]
    eval_df = pd.merge(actual_df, pred_df, on="date", how="inner")
    y_true  = eval_df["actual"].values; y_pred = eval_df["predicted"].values
    mape   = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    mae    = np.mean(np.abs(y_true - y_pred))
    rmse   = np.sqrt(np.mean((y_true - y_pred)**2))
    ss_res = np.sum((y_true - y_pred)**2)
    ss_tot = np.sum((y_true - y_true.mean())**2)
    r2     = 1 - (ss_res / ss_tot) if ss_tot > 0 else np.nan
    a_dir  = np.diff(y_true); p_dir = y_pred[1:] - y_true[:-1]
    da     = np.mean((a_dir > 0) == (p_dir > 0)) * 100 if len(a_dir) > 0 else np.nan
    y_train = full_ts[:split_idx].values().flatten()
    naive_mae = np.mean(np.abs(np.diff(y_train))) if len(y_train) > 1 else np.nan
    mase = mae / naive_mae if (not np.isnan(naive_mae) and naive_mae > 0) else np.nan
    if len(y_true) > 2 and len(a_dir) > 0:
        ret = np.diff(y_true) / y_true[:-1]; sigma = np.std(ret)
        lm  = np.abs(ret) > sigma
        hit_large = np.mean((a_dir[lm] > 0) == (p_dir[lm] > 0)) * 100 if lm.sum() > 0 else np.nan
    else: hit_large = np.nan
    return {"mape":round(mape,4),"mae":round(mae,4),"rmse":round(rmse,4),
            "r2":round(r2,4),"da":round(da,2),
            "mase":round(mase,4) if not np.isnan(mase) else np.nan,
            "hit_large":round(hit_large,2) if not np.isnan(hit_large) else np.nan}


def evaluate_model_default(model_name, grp_name, grp_vars, window, horizon):
    target_ts, cov_ts = to_series(df_merged, "IHSG", grp_vars if grp_vars else None)
    n         = len(target_ts)
    split_idx = int(n * TRAIN_RATIO)
    train_scaled, full_scaled, scaler = transform_target(target_ts, split_idx)
    full_cov_scaled = transform_covariates(cov_ts, split_idx)
    model = build_model_default(model_name, window, horizon, has_covariates=bool(grp_vars))
    model.fit(train_scaled, past_covariates=full_cov_scaled)
    test_start    = target_ts[split_idx].start_time()
    forecast_list = model.historical_forecasts(
        series=full_scaled, past_covariates=full_cov_scaled,
        start=test_start, forecast_horizon=horizon, stride=horizon,
        retrain=False, last_points_only=False, verbose=False)
    if isinstance(forecast_list, TimeSeries): forecast_list = [forecast_list]
    return inverse_and_metrics(forecast_list, target_ts, scaler, split_idx)

print("Helpers defined. Pipeline: DEFAULT params + 80/20 temporal split")

Helpers defined. Pipeline: DEFAULT params + 80/20 temporal split


In [11]:
MODEL_NAMES    = ["RandomForest", "ExtraTrees", "XGBoost", "LightGBM"]
default_results = []
default_failed  = []
exp_num         = 0
start_time      = datetime.now()

print(f"Started : {start_time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Total   : {total_exp} experiments  (DEFAULT params)")
print("=" * 82)

for model_name in MODEL_NAMES:
    for grp_name, grp_vars in GROUP_COVARIATES.items():
        for window in WINDOWS:
            for horizon in HORIZONS:
                exp_num += 1
                tag = (f"[{exp_num:>3}/{total_exp}] "
                       f"{model_name:13s} | {grp_name:22s} | W{window:>3}_H{horizon:>2}")
                print(f"{tag}", end=" ... ", flush=True)
                try:
                    m = evaluate_model_default(model_name, grp_name, grp_vars, window, horizon)
                    default_results.append({
                        "Model": model_name, "Covariates": grp_name,
                        "Window": window, "Horizon": horizon, **m,
                    })
                    print(f"MAPE={m['mape']:.4f}%  DA={m['da']:.1f}%  MASE={m['mase']:.4f}")
                except Exception as e:
                    default_failed.append({"tag": tag, "error": str(e)})
                    print(f"FAILED: {e}")
                    traceback.print_exc()
                finally:
                    gc.collect()

        # Auto-save checkpoint per model selesai satu group
        if default_results:
            pd.DataFrame(default_results).to_csv("phase1b_default_results.csv", index=False)
            print(f"  → Checkpoint saved ({len(default_results)} rows)")

elapsed = datetime.now() - start_time
print("=" * 82)
print(f"Done in {elapsed} | {len(default_results)} OK, {len(default_failed)} failed")

df_default = pd.DataFrame(default_results)
df_default.to_csv("phase1b_default_results.csv", index=False)
print(f"Saved: phase1b_default_results.csv ({len(df_default)} rows)")

Started : 2026-05-09 20:03:57
Total   : 144 experiments  (DEFAULT params)
[  1/144] RandomForest  | Baseline               | W 20_H 1 ... MAPE=0.5060%  DA=52.3%  MASE=0.9723
[  2/144] RandomForest  | Baseline               | W 20_H 5 ... MAPE=0.8523%  DA=47.7%  MASE=1.6376
[  3/144] RandomForest  | Baseline               | W 20_H20 ... MAPE=1.3849%  DA=49.5%  MASE=2.6727
[  4/144] RandomForest  | Baseline               | W120_H 1 ... MAPE=0.5057%  DA=52.1%  MASE=0.9717
[  5/144] RandomForest  | Baseline               | W120_H 5 ... MAPE=0.8526%  DA=47.9%  MASE=1.6380
[  6/144] RandomForest  | Baseline               | W120_H20 ... MAPE=1.3841%  DA=49.7%  MASE=2.6706
  → Checkpoint saved (6 rows)
[  7/144] RandomForest  | Screening1             | W 20_H 1 ... MAPE=0.5046%  DA=55.1%  MASE=0.9697
[  8/144] RandomForest  | Screening1             | W 20_H 5 ... MAPE=0.8520%  DA=47.9%  MASE=1.6373
[  9/144] RandomForest  | Screening1             | W 20_H20 ... MAPE=1.4130%  DA=49.1%  MASE=2.7

## Tabel Parameter: Default vs Optuna

Tabel ini menunjukkan parameter apa saja yang dioptimasi Optuna dan seberapa jauh berubah dari nilai default.

In [7]:
TUNING = joblib.load("saved_models/optuna_tuning_results.joblib")

# Semua parameter yang muncul di salah satu model
all_params = set()
for cfg in DEFAULT_CONFIGS.values():
    all_params.update(cfg.keys())
for r in TUNING.values():
    all_params.update(r['best_params'].keys())
all_params = sorted(all_params)

MODEL_ORDER = ["RandomForest","ExtraTrees","XGBoost","LightGBM"]
rows = []
for param in all_params:
    row = {"Parameter": param}
    for m in MODEL_ORDER:
        def_val  = DEFAULT_CONFIGS[m].get(param, "—")
        opt_val  = TUNING[m]['best_params'].get(param, "—")
        row[f"{m[:2]}_Default"] = def_val
        row[f"{m[:2]}_Optuna"]  = opt_val
    rows.append(row)

df_params = pd.DataFrame(rows).set_index("Parameter")

# Format angka float
def fmt_val(v):
    if isinstance(v, float) and abs(v) < 0.01:
        return f"{v:.2e}"
    if isinstance(v, float):
        return f"{v:.4f}"
    return str(v)

print("="*70)
print("  TABEL PARAMETER: Default vs Optuna (per Algoritma)")
print("="*70)

def highlight_changed(val, default_val):
    if val == "—" or default_val == "—": return ""
    try:
        if str(val) != str(default_val): return "background-color:#fff9c4;font-weight:bold"
    except: pass
    return ""

styled = df_params.copy()
for col in styled.columns:
    styled[col] = styled[col].apply(fmt_val)

display(
    styled.style
    .set_caption(
        "Kuning = parameter berubah dari default. "
        "RF=RandomForest, ET=ExtraTrees, XG=XGBoost, LG=LightGBM"
    )
)

# Tabel ringkas: Optuna best value vs default MAPE proxy
print("\n" + "="*60)
print("  Best Value Optuna (objective function) per Model")
print("="*60)
for m in MODEL_ORDER:
    print(f"  {m:15s}: Optuna best = {TUNING[m]['best_value']:.4f}  |  params = {TUNING[m]['best_params']}")

  TABEL PARAMETER: Default vs Optuna (per Algoritma)


,Ra_Default,Ra_Optuna,Ex_Default,Ex_Optuna,XG_Default,XG_Optuna,Li_Default,Li_Optuna
Parameter,,,,,,,,
colsample_bytree,—,—,—,—,0.7000,0.3920,0.7000,0.3455
learning_rate,—,—,—,—,0.1000,0.0295,0.1000,0.0117
max_depth,5,3,5,5,5,11,5,10
max_features,sqrt,0.7000,sqrt,sqrt,—,—,—,—
max_samples,0.7000,0.8000,—,—,—,—,—,—
min_child_samples,—,—,—,—,—,—,—,42
min_child_weight,—,—,—,—,—,8,—,—
min_samples_leaf,—,6,—,3,—,—,—,—
min_samples_split,—,5,—,9,—,—,—,—



  Best Value Optuna (objective function) per Model
  RandomForest   : Optuna best = 1.2634  |  params = {'n_estimators': 800, 'max_depth': 3, 'max_features': 0.7, 'max_samples': 0.8, 'min_samples_split': 5, 'min_samples_leaf': 6}
  ExtraTrees     : Optuna best = 1.2625  |  params = {'n_estimators': 200, 'max_depth': 5, 'max_features': 'sqrt', 'min_samples_split': 9, 'min_samples_leaf': 3}
  XGBoost        : Optuna best = 1.2589  |  params = {'n_estimators': 600, 'max_depth': 11, 'learning_rate': 0.029464057132418377, 'subsample': 0.8256085472109767, 'colsample_bytree': 0.39195610746628573, 'reg_alpha': 4.723006221405656, 'reg_lambda': 0.0034555486843382047, 'min_child_weight': 8}
  LightGBM       : Optuna best = 1.2611  |  params = {'n_estimators': 600, 'max_depth': 10, 'learning_rate': 0.011711509955524094, 'num_leaves': 83, 'subsample': 0.5852620618436457, 'colsample_bytree': 0.3455361150896956, 'reg_alpha': 3.4671276804481113, 'reg_lambda': 4.905556676028774, 'min_child_samples': 4

## Tabel Perbandingan Hasil: Default vs Optuna

Load kedua CSV dan tampilkan perbandingan langsung.

In [8]:
# Load kedua hasil
df_def = pd.read_csv("phase1b_default_results.csv")
df_opt = pd.read_csv("phase1a_group_results.csv")

MODEL_ORDER    = ["RandomForest","ExtraTrees","XGBoost","LightGBM"]
GROUP_ORDER    = ["Baseline","Screening1","Screening2",
                  "All_Commodity_STI","All_Macro_no_UST","All_Covariates"]
SCENARIO_ORDER = ["W20_H1","W120_H1","W20_H5","W120_H5","W20_H20","W120_H20"]

def add_scenario(df):
    df = df.copy()
    df["Skenario"] = "W" + df["Window"].astype(str) + "_H" + df["Horizon"].astype(str)
    return df

df_def = add_scenario(df_def)
df_opt = add_scenario(df_opt)

def make_comparison(df_d, df_o, groupby_col, order):
    """Buat tabel avg MAPE Default vs Optuna + improvement%."""
    d = df_d.groupby(groupby_col)["mape"].mean().reindex(order).rename("Default MAPE")
    o = df_o.groupby(groupby_col)["mape"].mean().reindex(order).rename("Optuna MAPE")
    cmp = pd.concat([d, o], axis=1)
    cmp["Impr % (Optuna vs Default)"] = ((cmp["Default MAPE"] - cmp["Optuna MAPE"])
                                          / cmp["Default MAPE"] * 100).round(3)

    d_da = df_d.groupby(groupby_col)["da"].mean().reindex(order).rename("Default DA")
    o_da = df_o.groupby(groupby_col)["da"].mean().reindex(order).rename("Optuna DA")
    cmp = pd.concat([cmp, d_da, o_da], axis=1)
    cmp["DA Impr (pp)"] = (cmp["Optuna DA"] - cmp["Default DA"]).round(2)
    return cmp.round({"Default MAPE":4,"Optuna MAPE":4,"Default DA":2,"Optuna DA":2})

def style_cmp(df):
    return (df.style
            .format({"Default MAPE":"{:.4f}","Optuna MAPE":"{:.4f}",
                     "Impr % (Optuna vs Default)":"{:+.3f}",
                     "Default DA":"{:.2f}","Optuna DA":"{:.2f}",
                     "DA Impr (pp)":"{:+.2f}"})
            .map(lambda v: "color:#2e7d32;font-weight:bold" if isinstance(v,float) and v>0
                 else ("color:#c62828" if isinstance(v,float) and v<0 else ""),
                 subset=["Impr % (Optuna vs Default)","DA Impr (pp)"])
            .highlight_min(subset=["Default MAPE","Optuna MAPE"], color="#c8e6c9"))

# ── Tabel 1: Per Model ────────────────────────────────────────────────────────
print("="*65)
print("  Tabel 1 — Avg MAPE & DA per Model: Default vs Optuna")
print("  (avg semua group × window × horizon)")
print("="*65)
display(style_cmp(make_comparison(df_def, df_opt, "Model", MODEL_ORDER))
        .set_caption("Tabel 1: Perbandingan per Algoritma ML"))

# ── Tabel 2: Per Group Covariate ─────────────────────────────────────────────
print("\n" + "="*65)
print("  Tabel 2 — Avg MAPE & DA per Group Covariate: Default vs Optuna")
print("  (avg semua model × window × horizon)")
print("="*65)
display(style_cmp(make_comparison(df_def, df_opt, "Covariates", GROUP_ORDER))
        .set_caption("Tabel 2: Perbandingan per Group Covariate"))

# ── Tabel 3: Per Skenario ─────────────────────────────────────────────────────
print("\n" + "="*65)
print("  Tabel 3 — Avg MAPE & DA per Skenario: Default vs Optuna")
print("  (avg semua model × group)")
print("="*65)
display(style_cmp(make_comparison(df_def, df_opt, "Skenario", SCENARIO_ORDER))
        .set_caption("Tabel 3: Perbandingan per Skenario (Window × Horizon)"))

  Tabel 1 — Avg MAPE & DA per Model: Default vs Optuna
  (avg semua group × window × horizon)


,Default MAPE,Optuna MAPE,Impr % (Optuna vs Default),Default DA,Optuna DA,DA Impr (pp)
Model,,,,,,
RandomForest,0.9169,0.9223,-0.594,50.35,50.11,-0.24
ExtraTrees,0.9145,0.9153,-0.087,50.00,50.05,+0.05
XGBoost,1.0075,0.9080,+9.873,49.74,49.59,-0.15
LightGBM,1.0438,0.9047,+13.330,49.69,49.70,+0.01



  Tabel 2 — Avg MAPE & DA per Group Covariate: Default vs Optuna
  (avg semua model × window × horizon)


,Default MAPE,Optuna MAPE,Impr % (Optuna vs Default),Default DA,Optuna DA,DA Impr (pp)
Covariates,,,,,,
Baseline,0.9548,0.9169,+3.967,49.61,49.63,+0.01
Screening1,0.9657,0.9047,+6.315,49.96,50.16,+0.20
Screening2,0.9768,0.9080,+7.038,50.05,50.06,+0.00
All_Commodity_STI,0.9763,0.9071,+7.084,49.97,49.89,-0.09
All_Macro_no_UST,0.9623,0.9185,+4.550,50.26,49.74,-0.53
All_Covariates,0.9881,0.9201,+6.884,49.83,49.71,-0.12



  Tabel 3 — Avg MAPE & DA per Skenario: Default vs Optuna
  (avg semua model × group)


,Default MAPE,Optuna MAPE,Impr % (Optuna vs Default),Default DA,Optuna DA,DA Impr (pp)
Skenario,,,,,,
W20_H1,0.5242,0.5049,+3.686,53.45,53.35,-0.10
W120_H1,0.5243,0.5063,+3.428,52.47,51.98,-0.49
W20_H5,0.8852,0.8423,+4.847,48.15,48.05,-0.10
W120_H5,0.8740,0.8468,+3.120,48.40,48.19,-0.21
W20_H20,1.5365,1.3950,+9.214,48.73,48.93,+0.20
W120_H20,1.4798,1.3802,+6.725,48.48,48.68,+0.19


In [9]:
# ── Tabel 4: Best & Worst per Model (Default vs Optuna) ───────────────────────
print("="*70)
print("  Tabel 4 — Best Configuration per Model: Default vs Optuna")
print("="*70)

rows = []
for model_name in MODEL_ORDER:
    for label, df_src in [("Default", df_def), ("Optuna", df_opt)]:
        sub = df_src[(df_src["Model"] == model_name) &
                     (df_src["Covariates"] != "Baseline")]
        if sub.empty: continue
        best  = sub.loc[sub["mape"].idxmin()]
        worst = sub.loc[sub["mape"].idxmax()]
        for typ, row in [("Best", best), ("Worst", worst)]:
            rows.append({
                "Model":   model_name,
                "Params":  label,
                "Type":    typ,
                "Group":   row["Covariates"],
                "Window":  int(row["Window"]),
                "Horizon": int(row["Horizon"]),
                "MAPE (%)":    round(row["mape"], 4),
                "DA (%)":      round(row["da"],   2),
            })

df_best = pd.DataFrame(rows)

def color_params(val):
    return {"Default":"background-color:#fff9c4","Optuna":"background-color:#c8e6c9"}.get(val,"")
def color_type(val):
    return {"Best":"font-weight:bold;color:#1b5e20","Worst":"color:#b71c1c"}.get(val,"")

display(
    df_best.style
    .map(color_params, subset=["Params"])
    .map(color_type,   subset=["Type"])
    .format({"MAPE (%)":"{:.4f}","DA (%)":"{:.2f}"})
    .set_caption("Tabel 4: Best & Worst config per Model — Kuning=Default, Hijau=Optuna")
)

# ── Ringkasan singkat ──────────────────────────────────────────────────────────
print("\n" + "="*65)
print("  RINGKASAN: Improvement Optuna vs Default")
print("="*65)
overall_def = df_def[df_def["Covariates"]!="Baseline"]["mape"].mean()
overall_opt = df_opt[df_opt["Covariates"]!="Baseline"]["mape"].mean()
impr_overall = (overall_def - overall_opt) / overall_def * 100
print(f"  Overall avg MAPE Default : {overall_def:.4f}%")
print(f"  Overall avg MAPE Optuna  : {overall_opt:.4f}%")
print(f"  Improvement              : {impr_overall:+.3f}%")
print()
for model_name in MODEL_ORDER:
    d_m = df_def[df_def["Model"]==model_name]["mape"].mean()
    o_m = df_opt[df_opt["Model"]==model_name]["mape"].mean()
    impr = (d_m - o_m) / d_m * 100
    print(f"  {model_name:15s}: Default={d_m:.4f}%  Optuna={o_m:.4f}%  Impr={impr:+.3f}%")

  Tabel 4 — Best Configuration per Model: Default vs Optuna


,Model,Params,Type,Group,Window,Horizon,MAPE (%),DA (%)
0,RandomForest,Default,Best,Screening1,20,1,0.5046,55.13
1,RandomForest,Default,Worst,Screening1,20,20,1.4130,49.13
2,RandomForest,Optuna,Best,Screening1,20,1,0.5047,53.80
3,RandomForest,Optuna,Worst,All_Covariates,20,20,1.4750,49.71
4,ExtraTrees,Default,Best,Screening2,20,1,0.5049,53.80
5,ExtraTrees,Default,Worst,Screening1,20,20,1.3943,48.55
6,ExtraTrees,Optuna,Best,Screening2,120,1,0.5047,54.75
7,ExtraTrees,Optuna,Worst,All_Covariates,20,20,1.4007,48.55
8,XGBoost,Default,Best,Screening1,20,1,0.5314,56.08
9,XGBoost,Default,Worst,All_Covariates,20,20,1.7228,49.52



  RINGKASAN: Improvement Optuna vs Default
  Overall avg MAPE Default : 0.9739%
  Overall avg MAPE Optuna  : 0.9117%
  Improvement              : +6.381%

  RandomForest   : Default=0.9169%  Optuna=0.9223%  Impr=-0.594%
  ExtraTrees     : Default=0.9145%  Optuna=0.9153%  Impr=-0.087%
  XGBoost        : Default=1.0075%  Optuna=0.9080%  Impr=+9.873%
  LightGBM       : Default=1.0438%  Optuna=0.9047%  Impr=+13.330%
